In [61]:
from transformers import ViTImageProcessor, ViTModel, ViTConfig
from transformers.utils import cached_file
import torch.nn as nn
import torch 
import glob
import random
from PIL import Image
import json
import torchvision.transforms as T

from transformers import ViTImageProcessor, ViTModel, ViTConfig
from transformers.utils import cached_file
import torch
from PIL import Image
import requests
import timm
import lightning as L

import sys
sys.path.append("../../../../donut/src/test/")
from common.config import cfg

from image_recog_datamodule import ImageClsDataModule
from image_recog_datasets import ImageClsDataset
from class_weighting import compute_class_weights
from lightning.pytorch import Trainer
from lightning.pytorch.loggers import MLFlowLogger
from datetime import datetime

import sys
sys.path.append("../../../../donut/src/test/")
from common.config import cfg

In [62]:
# load variables

local_config_path = cfg.vit_local_config_path

# label encoder
label_encoder = {
    "dot": cfg.dot,
    "scatter": cfg.scatter,
    "horizontal_bar": cfg.horizontal_bar,
    "line": cfg.line,
    "vertical_bar": cfg.vertical_bar,
}

# label decoder
label_decoder = {v: k for k, v in label_encoder.items()}

id2label = {idx: label for idx, label in label_decoder.items()}
label2id = {label: idx for idx, label in label_encoder.items()}

# Dataset parameters
image_cls_height = cfg.vit_image_cls_height
image_cls_width = cfg.vit_image_cls_width

In [63]:
config = ViTConfig.from_pretrained("google/vit-base-patch16-224-in21k",cache_dir=local_config_path)

In [64]:
# add customized paramaters into config file

config.num_labels = len(id2label)
config.id2label = id2label
config.label2id = label2id
config.hidden_dropout_prob = 0.1 
config.image_size = 320
config.patch_size = 16
# 320/16=20,
# input token num: 320*320 / 16*16 = 20*20=400 
# add CLS: 400+1=401
# each token seq len: 16*16*3=768

model_name = "google/vit-base-patch16-224-in21k"
config_path = cached_file(model_name, "config.json")

In [65]:
import torch
from transformers import ViTModel, ViTConfig
from torch import nn


class SimpleViTClassifier(nn.Module):
    def __init__(self, config, num_classes):
        super().__init__()
        self.vit = ViTModel(config)
        self.classifier = nn.Linear(config.hidden_size, num_classes)

    def forward(self, x):
        outputs = self.vit(x)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_token)

# load raw state dictionary
raw_state_dict = torch.load("../../../trained_models/vit-best-epoch=12-val_acc=0.9985.pt", map_location="cpu")

# keymapping:  "model." → "vit."
converted_state_dict = {}
for k, v in raw_state_dict.items():
    new_key = k.replace("model.", "vit.") if k.startswith("model.") else k
    converted_state_dict[new_key] = v


model = SimpleViTClassifier(config=config, num_classes=len(id2label))
model.load_state_dict(converted_state_dict)
model.eval()
model.to("cpu")
model_device = next(model.parameters()).device 
print(model_device)

cpu


In [66]:
# select images

# list all image paths
image_dir = "../../../data/image_resize/images/"
image_files = glob.glob(image_dir + "*")
print(image_files[:5])

# list all annotation files paths

annotations_dir="../../../data/image_resize/annotations/"
annotations_files=glob.glob(annotations_dir+"*")
print(annotations_files[:5])


# select 20 pics randomly

selected_imgs= random.sample(image_files, 20)
print(selected_imgs)

# extract the related labels
selected_labels=[]
for idx in range(len(selected_imgs)):
    img_path=selected_imgs[idx]
    image_id=img_path.split('/')[-1].split('.')[0]
    anno_path = [file for file in annotations_files if image_id in file]
    with open(anno_path[0],"r") as f:
        annottation=json.load(f)
        selected_labels.append(annottation["chart-type"])



['../../../data/image_resize/images/45df1fe3293b.jpg', '../../../data/image_resize/images/b2ab3b743d4e.jpg', '../../../data/image_resize/images/51d3b1a6baf3.jpg', '../../../data/image_resize/images/a9e9ce9277c1.jpg', '../../../data/image_resize/images/7f1f545fe081.jpg']
['../../../data/image_resize/annotations/e91e28111e86.json', '../../../data/image_resize/annotations/75c0449f6917.json', '../../../data/image_resize/annotations/66dd2a250237.json', '../../../data/image_resize/annotations/58595c30beab.json', '../../../data/image_resize/annotations/497a547454d7.json']
['../../../data/image_resize/images/9ff78aa5ebd9.jpg', '../../../data/image_resize/images/019f7a52d588.jpg', '../../../data/image_resize/images/b8f5e4e28d9e.jpg', '../../../data/image_resize/images/6c8e4dcc75ec.jpg', '../../../data/image_resize/images/c37a6d6ebdd4.jpg', '../../../data/image_resize/images/3e8f8a799e86.jpg', '../../../data/image_resize/images/9c04845dae08.jpg', '../../../data/image_resize/images/a27bdadc220b.j

In [67]:
transform = T.Compose([
    T.Resize((image_cls_height, image_cls_width)),                    
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
])

results=[]
for img in selected_imgs:
    img = Image.open(img).convert("RGB")
    input_tensor = transform(img).unsqueeze(0).to(model_device)

    with torch.no_grad():
        logits = model(input_tensor)
        pred = torch.argmax(logits, dim=1).item()

    results.append(pred)

In [68]:
predicted_labels = [config.id2label[id] for id in results]

In [69]:
predicted_labels

['vertical_bar',
 'line',
 'dot',
 'line',
 'vertical_bar',
 'scatter',
 'vertical_bar',
 'vertical_bar',
 'vertical_bar',
 'vertical_bar',
 'vertical_bar',
 'dot',
 'vertical_bar',
 'scatter',
 'line',
 'line',
 'line',
 'dot',
 'vertical_bar',
 'scatter']

In [70]:
matches = [a == b for a, b in zip(selected_labels, predicted_labels)]

# calculate the accuracy
accuracy = sum(matches) / len(matches)

print(f"accuracy: {accuracy}")

accuracy: 1.0


In [71]:
print(matches)

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]


In [72]:
# # self prepared data

# image_dir = "../../../data/prediction-data-self-create/generated_charts/"
# image_files = glob.glob(image_dir + "*")
# print(image_files[:5])


# # extract the related labels
# selected_labels=[]
# for idx in range(len(image_files)):
#     img_path=image_files[idx]
#     image_type=img_path.split('/')[-1].split('.')[0].split('-')[0]
#     selected_labels.append(image_type)

# print(selected_labels)

In [73]:
# transform = T.Compose([
#     T.Resize((image_cls_height, image_cls_width)),                    
#     T.ToTensor(),
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
# ])

# results=[]
# for img in image_files:
#     img = Image.open(img).convert("RGB")
#     input_tensor = transform(img).unsqueeze(0).to(model_device)

#     with torch.no_grad():
#         logits = model(input_tensor)
#         pred = torch.argmax(logits, dim=1).item()

#     results.append(pred)

In [74]:
# predicted_labels = [config.id2label[id] for id in results]

In [75]:
# predicted_labels

In [76]:
# matches = [a == b for a, b in zip(selected_labels, predicted_labels)]

# # calculate the accuracy
# accuracy = sum(matches) / len(matches)

# print(f"accuracy: {accuracy}")

In [77]:
# print(matches)

In [78]:
# indexes = [i for i, val in enumerate(matches) if val is False]
# print(indexes)

In [79]:
# elements = [image_files[i] for i in indexes]

# elements

In [80]:

# prediction_err = [predicted_labels[i] for i in indexes]

# prediction_err